In [ ]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
from pathlib import Path

from pyhealth.tasks.georgia_ecg import load_georgia_data

np.random.seed(1)
torch.manual_seed(1)
DATA_ROOT = Path("/content/data") if Path("/content/data").exists() else Path("data")
PROCESSED_ROOT = DATA_ROOT / "georgia_processed"
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
train, val, test, codes = load_georgia_data(
    data_root=DATA_ROOT, processed_root=PROCESSED_ROOT, seed=1
)
val_loader = DataLoader(val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0) #validation data loader
first_batch = next(iter(val_loader))
print("Splits:", len(train), len(val), len(test), "classes:", len(codes))
print("Validation batch:", first_batch["signal"].shape, first_batch["labels"].shape)
assert first_batch["signal"].shape == (32, 8, 12, 1250)
assert first_batch["labels"].shape == (32, 30)

In [ ]:
#TODO
# this is a temporary dummy model for testing, should be replaced
class DummyModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.linear = nn.Linear(12, num_classes)

    def forward(self, signal):
        # [B, 12, 1250] -> [B, 12] -> [B, 30] logits
        return self.linear(signal.mean(dim=-1))

model = DummyModel(len(codes)).to(device)
print("Dummy model output:", model(torch.zeros(2, 12, 1250, device=device)).shape)


In [ ]:
def predict_windows(model, signal, device):
    # [B, W, 12, 1250] -> [B*W, 12, 1250] -> [B, W, 30]
    batch_size, windows = signal.shape[:2]
    flat_signal = signal.reshape(batch_size * windows, 12, 1250).to(device)
    model.eval()
    with torch.no_grad():
        output = model(flat_signal)
        if isinstance(output, dict):
            # PyHealth may return probabilities directly; do not apply sigmoid twice.
            if "y_prob" in output:
                probabilities = output["y_prob"]
            else:
                probabilities = torch.sigmoid(output["logits"])
        else:
            probabilities = torch.sigmoid(output)
    return probabilities.reshape(batch_size, windows, len(codes)).cpu()

all_probabilities = []
all_labels = []
record_ids = []
for batch in val_loader:
    all_probabilities.append(predict_windows(model, batch["signal"], device))
    all_labels.append(batch["labels"].cpu())
    record_ids.extend(batch["record_id"])

window_probabilities = torch.cat(all_probabilities)
labels = torch.cat(all_labels).numpy()
print("Window probabilities:", window_probabilities.shape)
print("Labels:", labels.shape, "record IDs:", len(record_ids))


In [ ]:
def aggregate_predictions(probabilities):
    return {
        "Max": probabilities.max(dim=1).values,
        "Mean": probabilities.mean(dim=1),
        "Top-2 Mean": probabilities.topk(2, dim=1).values.mean(dim=1),
    }

aggregated = aggregate_predictions(window_probabilities)
for name, values in aggregated.items():
    print(name, values.shape)


In [ ]:
def evaluate_auc(y_true, probabilities, codes):
    scores = np.asarray(probabilities)
    per_class = pd.Series(
        [roc_auc_score(y_true[:, i], scores[:, i]) for i in range(len(codes))],
        index=codes, name="ROC-AUC"
    )
    return per_class.mean(), per_class

metrics = {}
for name, values in aggregated.items():
    metrics[name] = evaluate_auc(labels, values.numpy(), codes)


In [ ]:
# Small numerical checks before interpreting the full validation run.
toy = torch.tensor([[[0.1, 0.9], [0.8, 0.2], [0.6, 0.7]]])
toy_results = aggregate_predictions(toy)
assert torch.allclose(toy_results["Max"], torch.tensor([[0.8, 0.9]]))
assert torch.allclose(toy_results["Mean"], torch.tensor([[0.5, 0.6]]))
assert torch.allclose(toy_results["Top-2 Mean"], torch.tensor([[0.7, 0.8]]))
assert all(value.shape == (1, 2) for value in toy_results.values())
toy_labels = np.array([[0, 1], [1, 0], [0, 1], [1, 0]])
toy_scores = np.array([[0.1, 0.9], [0.9, 0.1], [0.2, 0.8], [0.8, 0.2]])
assert evaluate_auc(toy_labels, toy_scores, ["a", "b"])[0] == 1.0

assert len(val) == len(record_ids) == 1038
assert window_probabilities.shape == (1038, 8, 30)
assert labels.shape == (1038, 30)
assert len(set(record_ids)) == 1038
assert record_ids == [val[i]["record_id"] for i in range(len(val))]
assert np.isin(labels, [0, 1]).all()
assert torch.isfinite(window_probabilities).all()
assert ((window_probabilities >= 0) & (window_probabilities <= 1)).all()
for name, values in aggregated.items():
    assert values.shape == (1038, 30)
    assert torch.isfinite(values).all()
    assert np.isfinite(metrics[name][0])
    assert len(metrics[name][1]) == 30
print("PASS: manual aggregation/AUC checks and all 1,038 validation records")


In [ ]:
results = pd.DataFrame({
    "Aggregation": list(metrics),
    "Macro ROC-AUC": [metrics[name][0] for name in metrics],
})
results["Delta vs. Max"] = results["Macro ROC-AUC"] - results.loc[0, "Macro ROC-AUC"]
print("Dummy Model Validation Results — pipeline check only; not paper reproduction")
print(results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("Per-class ROC-AUC available in metrics[name][1], indexed by SNOMED code.")
